In [1]:
from nltk.corpus import movie_reviews

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

In [3]:
# 사용자 정의 토크나이저
class SimpleTokenizer:
    def __init__(self, num_words = 10000, oov_token='UNK'):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}
        self.index_word = {}

    def _clean_text(self, text):
        return ' '.join(re.findall(r'\w+', text)) # 또는 re.sub(r'[^\w\s]', '', text).strip()
    
    def fit_on_texts(self, texts):
        '''빈도순으로 상위 단어 추출 토큰을 숫자로 변경, 공백을 기준으로 토큰 분류'''
        word_counts = Counter()

        for text in texts:
            word_counts.update(self._clean_text(text))

            # 빈도순으로 num_words 단어 추출
            most_common = word_counts.most_common(self.num_words-2) # pad UNK 특수 토큰 자리 남기기

            # 0: padding 1: oov
            self.word_index = {self.oov_token: 1}
            for i, (word, _) in enumerate(most_common):
                self.word_index[word] = i + 2
            
            self.index_word = {idx: w for w, idx in self.word_index.items()}

    def texts_to_sequence(self, texts):
        sequence = []

        for text in texts:
            seq = []
            for word in self._clean_text(text):
                seq.append(self.word_index.get(word,1))
            sequence.append(seq)

        return sequence
    
def pad_sequence(sequences, maxlen, padding='pre', truncating='pre'):

    features = np.zeros((len(sequences), maxlen), dtype=int)

    for i, seq in enumerate(sequences):
        if len(seq) > maxlen:
            if truncating == 'pre':
                features[i,:] = seq[-maxlen:]

            else:
                features[i,:] = seq[:maxlen]
        
        else:
            if padding == 'pre':
                features[i,-len(seq):] = seq
            
            else:
                features[i,:len(seq)] = seq
         
    return features

In [4]:
texts = ['i love you', 'i like music', 'love you']
sim = SimpleTokenizer()
sim.texts_to_sequence(texts)

[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1]]

In [5]:
# 리뷰 데이터로 적용해서 오류 없는지 확인 및 수정
reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]

In [6]:
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts)
requens = sim.texts_to_sequence(texts)

In [7]:
features = pad_sequence(requens,500)

In [8]:
features.shape

(2, 500)

In [9]:
# RNN 모델
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        x = self.embedding(x)
        _, hn = self.rnn(x) # output, hn
        # output: 모든 시점(time-step)의 숨겨진 상태: 각 단어(시점)를 거칠때마다 계산된 모든 hidden state를 모아놓음
        # seq2seq 모델은 각 단어마다 결과를 내야하는 개체명 인식(NER)

        # hn: 마지막 시점의 상태; 전체 문장을 다 읽고 최종적으로 요약한 정보: 문서분류
        return self.fc(hn.squeeze(0))
    

In [10]:
# 테스트
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts) # 문자 -> 숫자
requens = sim.texts_to_sequence(texts) # 길이를 맞춤
features = pad_sequence(requens,500)
features = torch.LongTensor(features)
print(features.shape)
features = nn.Embedding(500,32)(features)
outputs, hn = nn.RNN(32,64,batch_first=True)(features)
outputs.shape, hn.shape

torch.Size([2, 500])


(torch.Size([2, 500, 64]), torch.Size([1, 2, 64]))

In [11]:
# 데이터를 가져오기
# x, y 분할
# 토크나이저 + pad_sequence --> 문자를 수자로 변환

# train, test split
# TorchTensor 변환
# TensorDataset --> Dataloader

# 모델 생성
# 옵티마이저
# 손실 함수 정의

In [12]:
fileids = movie_reviews.fileids()
reviews = [movie_reviews.raw(fileid) for fileid in fileids]
categories = [movie_reviews.categories(fileid)[0] for fileid in fileids]

max_words = 10000
maxlen = 500
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = SimpleTokenizer(num_words=max_words)
tokenizer.fit_on_texts(reviews)
X = tokenizer.texts_to_sequence(reviews)
X = pad_sequence(X, maxlen=maxlen)

label_dict = {'pos': 1, 'neg': 0}
y = np.array([label_dict[c] for c in categories])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

X_train_t = torch.LongTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.LongTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print('Preprocessing complete. X_train shape:', X_train.shape)

Preprocessing complete. X_train shape: (1600, 500)


In [13]:
def train_model(model, loader, optimizer, criterion, epochs=10):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += ((outputs > 0.5).float() == y_batch).sum().item()
        print(f'Epoch {epoch+1}: Loss {total_loss/len(loader):.4f}, Acc {correct/len(loader.dataset):.4f}')

model_dense = RNNModel(max_words, 32, maxlen).to(device)
criterion = nn.BCELoss()
optimizer_dense = optim.RMSprop(model_dense.parameters(), lr=0.001)
train_model(model_dense, train_loader, optimizer_dense, criterion)

Epoch 1: Loss 0.7891, Acc 0.4894
Epoch 2: Loss 0.7110, Acc 0.5112
Epoch 3: Loss 0.7064, Acc 0.4888
Epoch 4: Loss 0.7034, Acc 0.4994
Epoch 5: Loss 0.7048, Acc 0.5019
Epoch 6: Loss 0.7039, Acc 0.4844
Epoch 7: Loss 0.6993, Acc 0.4956
Epoch 8: Loss 0.6980, Acc 0.5262
Epoch 9: Loss 0.7078, Acc 0.4919
Epoch 10: Loss 0.7007, Acc 0.5138


In [16]:
# 평가
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()
            correct += ((outputs > 0.5).float() == y_batch).sum().item()
    return correct / len(loader.dataset)

print(f'Final Test Accuracy (model_dense): {evaluate(model_dense, test_loader):.4f}')

Final Test Accuracy (model_dense): 0.5125


### 장기 의존성(Long-Term Dependency)
- 문장이 길어지면 앞부분의 정보를 잊어버린다

In [17]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(BiLSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x) # output: 모든 시점의 정보이지만 여기서는 문서 전체에 대한 요약본 필요하기에 무시
       
        # hn: 마지막 시점의 상태(양방향)
            # 정방향: 정방향으로 문장을 끝까지 읽었을 때 마지막 상태
            # 역방향: 역방향으로 문장을 끝까지 읽었을 때 마지막 상태
        
        # cn: cell state(장기 기억); 분류를 위해서는 출력에 사용하지 않음

        # 마지막 타임스텝의 정방향/역방향 은닉 상태 결합
        
        x = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
        return self.fc(x)

model_bilstm = BiLSTMModel(max_words, 64, 64).to(device)
optimizer_bilstm = optim.Adam(model_bilstm.parameters(), lr=1e-4)
train_model(model_bilstm, train_loader, optimizer_bilstm, criterion, epochs=8)

Epoch 1: Loss 0.6951, Acc 0.4969
Epoch 2: Loss 0.6939, Acc 0.4969
Epoch 3: Loss 0.6929, Acc 0.4969
Epoch 4: Loss 0.6921, Acc 0.4988
Epoch 5: Loss 0.6914, Acc 0.5150
Epoch 6: Loss 0.6906, Acc 0.5331
Epoch 7: Loss 0.6899, Acc 0.5550
Epoch 8: Loss 0.6892, Acc 0.5769
